<a href="https://colab.research.google.com/github/lama-byte/ML_for_cybersecurity/blob/main/EDA%20%26%20Feature%20Engineering/All_Feat_PDF_EDA_ONLY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
pdf_df = pd.read_csv('/content/PDF_All_features.csv')

print ("Shape:", pdf_df.shape)
print("\nColumn names:")
print(pdf_df.columns.tolist())
print("\nData Columns:")
print(pdf_df.info())

pdf_df.head()

Shape: (19296, 42)

Column names:
['file_path', 'file_size', 'title_chars', 'encrypted', 'metadata_size', 'page_count', 'valid_pdf_header', 'image_count', 'text_length', 'object_count', 'font_object_count', 'embedded_file_count', 'average_embedded_file_size', 'stream_count', 'endstream_count', 'average_stream_size', 'entropy_of_streams', 'xref_count', 'xref_entries', 'name_obfuscations', 'total_filters', 'nested_filter_objects', 'objstm_count', 'js_count', 'javascript_count', 'uri_count', 'uses_nonstandard_port', 'action_count', 'aa_count', 'openaction_count', 'launch_count', 'submitform_count', 'acroform_count', 'xfa_count', 'jbig2decode_count', 'colors_count', 'richmedia_count', 'trailer_count', 'startxref_count', 'has_multiple_behavioral_keywords_in_one_object', 'used_ocr', 'label']

Data Columns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19296 entries, 0 to 19295
Data columns (total 42 columns):
 #   Column                                          Non-Null Count  Dtype  
--

,file_path,file_size,title_chars,encrypted,metadata_size,page_count,valid_pdf_header,image_count,text_length,object_count,...,acroform_count,xfa_count,jbig2decode_count,colors_count,richmedia_count,trailer_count,startxref_count,has_multiple_behavioral_keywords_in_one_object,used_ocr,label
0,/content/drive/MyDrive/UNB/Mastercard project/...,20362,8,0,254,5,1,0,8018,120,...,0,0,0,0,0,0,0,0,0,0
1,/content/drive/MyDrive/UNB/Mastercard project/...,28848,35,0,242,4,1,0,11568,36,...,0,0,0,0,0,0,0,0,0,0
2,/content/drive/MyDrive/UNB/Mastercard project/...,76563,12,0,138,1,1,1,1377,21,...,0,0,0,0,0,0,0,0,1,0
3,/content/drive/MyDrive/UNB/Mastercard project/...,67982,8,0,278,4,1,0,30607,46,...,1,0,0,0,0,0,0,0,0,0
4,/content/drive/MyDrive/UNB/Mastercard project/...,3997,37,0,156,1,1,0,1,10,...,0,0,0,0,0,0,0,1,1,1


In [ ]:
pdf_df.describe().T

,count,mean,std,min,25%,50%,75%,max
file_size,19296.0,71308.369818,418699.216515,24.0,9620.0,17462.000000,77876.500000,3.677767e+07
title_chars,19296.0,14.792548,19.411976,0.0,0.0,8.000000,25.000000,2.890000e+02
encrypted,19296.0,0.004353,0.065837,0.0,0.0,0.000000,0.000000,1.000000e+00
metadata_size,19296.0,150.339138,149.862796,0.0,0.0,137.000000,251.000000,2.560000e+03
page_count,19296.0,3.232017,9.918278,0.0,0.0,1.000000,2.000000,5.120000e+02
valid_pdf_header,19296.0,0.611578,0.487404,0.0,0.0,1.000000,1.000000,1.000000e+00
image_count,19296.0,1.389096,9.600749,0.0,0.0,0.000000,0.000000,5.920000e+02
text_length,19296.0,8761.265133,37921.556310,0.0,0.0,1195.000000,6292.500000,3.415343e+06
object_count,19296.0,270.254042,5074.942888,0.0,0.0,20.000000,70.000000,2.000060e+05
font_object_count,19296.0,2.025601,6.618683,0.0,0.0,0.000000,3.000000,7.940000e+02


In [ ]:
# Class balance
print(pdf_df['label'].value_counts())
print(pdf_df['label'].value_counts(normalize=True))

# Missing values
print("\nMissing values per column:")
print(pdf_df.isnull().sum()[pdf_df.isnull().sum() > 0])

# Exact duplicate rows (excluding file_path, since paths are always unique)
feature_cols = pdf_df.columns.drop(['file_path', 'label'])
dupe_count = pdf_df.duplicated(subset=feature_cols).sum()
print(f"\nExact duplicate feature rows: {dupe_count}")

label
1    9999
0    9297
Name: count, dtype: int64
label
1    0.51819
0    0.48181
Name: proportion, dtype: float64

Missing values per column:
Series([], dtype: int64)

Exact duplicate feature rows: 3943


In [ ]:
pdf_df.nunique()

,0
file_path,19296
file_size,13729
title_chars,135
encrypted,2
metadata_size,562
page_count,108
valid_pdf_header,2
image_count,104
text_length,6610
object_count,1342


In [ ]:
feature_cols = pdf_df.columns.drop(['file_path', 'label'])

# 1. Fully dead columns: only ONE unique value across all rows
dead_cols = [col for col in feature_cols if pdf_df[col].nunique() == 1]

print(f"Fully dead columns ({len(dead_cols)}):")
for col in dead_cols:
    print(f"  {col}  -> constant value: {pdf_df[col].unique()[0]}")

# 2. Near-dead columns: extremely low variance (e.g. >99% of rows share the same value)
print("\nNear-dead columns (>99% same value):")
for col in feature_cols:
    top_freq = pdf_df[col].value_counts(normalize=True).iloc[0]
    if top_freq > 0.99 and col not in dead_cols:
        print(f"  {col}  -> {top_freq:.2%} of rows are '{pdf_df[col].mode()[0]}'")

Fully dead columns (8):
  embedded_file_count  -> constant value: 0
  average_embedded_file_size  -> constant value: 0
  xref_count  -> constant value: 0
  xref_entries  -> constant value: 0
  submitform_count  -> constant value: 0
  jbig2decode_count  -> constant value: 0
  trailer_count  -> constant value: 0
  startxref_count  -> constant value: 0

Near-dead columns (>99% same value):
  encrypted  -> 99.56% of rows are '0'
  uses_nonstandard_port  -> 99.56% of rows are '0'
  launch_count  -> 99.58% of rows are '0'
  richmedia_count  -> 99.97% of rows are '0'


In [ ]:
near_dead = ['encrypted', 'uses_nonstandard_port', 'launch_count',
             'richmedia_count']

for col in near_dead:
    print(f"\n{col}:")
    print(pd.crosstab(pdf_df[col] > 0, pdf_df['label']))

# False = the value is exactly 0
# True = the value is anything above 0 (nonzero)


encrypted:
label         0     1
encrypted            
False      9220  9992
True         77     7

uses_nonstandard_port:
label                     0     1
uses_nonstandard_port            
False                  9215  9996
True                     82     3

launch_count:
label            0     1
launch_count            
False         9284  9930
True            13    69

richmedia_count:
label               0     1
richmedia_count            
False            9297  9993
True                0     6


In [ ]:
feature_cols = pdf_df.columns.drop(['file_path', 'label'])

# All rows involved in ANY duplication (keep=False marks every copy, not just the extras)
dupe_rows = pdf_df[pdf_df.duplicated(subset=feature_cols, keep=False)]
print("Total rows involved in duplicate groups:", dupe_rows.shape[0])

# How many distinct duplicate "groups" exist, and how large are they?
group_sizes = dupe_rows.groupby(list(feature_cols)).size() # -> each duplicate pattern is one group so it's like a list of the different groups with their sizes (the last col)
#  That means each "group" is identified not by one value, but by a combination of ~48 values

print("\nNumber of distinct duplicate groups:", len(group_sizes))
print("Group size distribution:") # calculates how many groups have the same size?
print(group_sizes.value_counts().sort_index())

# Class balance within duplicated rows
print("\nLabel distribution among duplicated rows:")
print(dupe_rows['label'].value_counts())

Total rows involved in duplicate groups: 5703

Number of distinct duplicate groups: 1760
Group size distribution:
2     889
3     306
4     219
5     148
6      96
7      49
8      26
9      15
10      8
11      1
12      2
14      1
Name: count, dtype: int64

Label distribution among duplicated rows:
label
1    5675
0      28
Name: count, dtype: int64


In [ ]:
# Show file_paths for a couple of duplicate groups so we can see what's actually happening
sample_group = dupe_rows.sort_values(by=list(feature_cols)).head(20)
# .sort_values -> sorting by all feature columns means identical rows will naturally land right next to each other in the output.


sample_group[['file_path', 'label', 'title_chars', 'metadata_size', 'file_size', 'entropy_of_streams']]

,file_path,label,title_chars,metadata_size,file_size,entropy_of_streams
6060,/content/drive/MyDrive/UNB/Mastercard project/...,1,0,0,1282,0.0
10306,/content/drive/MyDrive/UNB/Mastercard project/...,1,0,0,1282,0.0
4196,/content/drive/MyDrive/UNB/Mastercard project/...,1,0,0,1426,0.0
12374,/content/drive/MyDrive/UNB/Mastercard project/...,1,0,0,1426,0.0
3505,/content/drive/MyDrive/UNB/Mastercard project/...,1,0,0,1428,0.0
9310,/content/drive/MyDrive/UNB/Mastercard project/...,1,0,0,1428,0.0
16295,/content/drive/MyDrive/UNB/Mastercard project/...,1,0,0,1430,0.0
17938,/content/drive/MyDrive/UNB/Mastercard project/...,1,0,0,1430,0.0
18611,/content/drive/MyDrive/UNB/Mastercard project/...,1,0,0,1430,0.0
1952,/content/drive/MyDrive/UNB/Mastercard project/...,1,0,0,1432,0.0
